### Mini Project
- In this project, we will see if our linux server is running low on memory.
- If it is running low on memory then we will create a jira ticket.

In [ ]:
### ACCOUNT IDs of users
### you can get account ID byinspecting an issue created by the user  in json viewer and look for creator field
# ahsanzafar0501 = "712020:a16b5f52-74ef-4c10-a148-bfd621331a41"
# ahsanzafarqau = 712020:1e81b910-027d-4b0e-bfd0-9053aa482579


# jira api for finding user information through user id
# https://ahsanzafarqau.atlassian.net/rest/api/3/user?accountId=712020:1e81b910-027d-4b0e-bfd0-9053aa482579

In [8]:
import time
import paramiko
# lets import jira
from jira import JIRA
# importing smtp module
import smtplib

In [3]:
# linux server ip address
linux_server = "192.168.43.26"

# username
username = "ahsan-zafar"

# password
password = "AhsanHacker?1122?"

# creating client
client = paramiko.SSHClient()

# setting the policy
client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

# connecting with the client
client.connect(hostname=linux_server,username=username,password=password)

cmd = "free -g"

stdin, stdout, stderr = client.exec_command(cmd)




In [4]:
result = stdout.readlines()



In [7]:
# we need to check availble memory
available_memory = int(result[1].split()[-1])

In [15]:
if available_memory <= 2:
    # email of account through which we login
    email = "ahsanzafarqau@gmail.com"

    # api token of account
    api_token = "ATATT3xFfGF0rv4nzQuXRBzq_DLrL2aoMoURxzdQgFrbGxrvLsx-Pz2nDLJjSUuFO_4PxGzrRSvIMKgozMnD9mDd-IPlvFwoqoV7F4sBfWF3F20NNLDcGKcUur3gpk-wnDj8UJWMvW3H52qvCzfSm8WyjT-E4QU2iUqtD-Y5I5SRu35Sby4x7_0=757B7A9A"

    # website where we want to log in
    server = "https://ahsanzafarqau.atlassian.net"
    # now making a connection
    jira_connection = JIRA(server=server,basic_auth=(email,api_token))

    # details of issue we want to create
    issue_description = {
        'project':{'key':"LP"},
        'issuetype':{'name':'Task'},
        'description':f'The memory of server {linux_server} is {available_memory}GB and has dropped then the healthy threshold which is 2GB',
        'summary':f'Low memory on sever {linux_server}',
        'priority':{'name':'Highest'},
        
        # some projects does not have timetracking field by default, go to screen and add timetracking field for that screen
        'timetracking':{'originalEstimate':"1h"}
    }

    # method # 1
    issue = jira_connection.create_issue(issue_description)

    # assigning the ticket to the ahsanzafar0501 use
    issue.update(assignee={"accountId":"712020:a16b5f52-74ef-4c10-a148-bfd621331a41"})


    ## Part two of project, we need to send mail to the person to whom ticket is assigned and we need to send mail to the person who created the mail

    # issue key
    issue_key = issue.key

    # finding details of issue
    issue_details = jira_connection.issue(id=issue_key)

    # finding the reporter of issue
    issue_reporter = issue_details.fields.reporter.displayName

    # finding email of reporter
    issue_reporter_email = issue_details.fields.reporter.emailAddress

    # defining the email from which mails will be sent to issue reporters
    my_mail = "ahsanzafarqau@gmail.com"


    # mail to which message will be send, as we are iterating on tickets, so different ticket may be reported
    # by different individuals so using the index to keep track of information
    to_mail = issue_reporter_email

        # my app password set in my google account ahsanzafarqau@gmail.com
    my_pswd = "dqrhjavpyrejiaza"

        # you need to define subject and message in capital letters
    SUBJECT=issue_description["summary"]
        
    # creating the message that the ticket creator will see
    MESSAGE = f"""Hi {issue_reporter}, thanks for reporting us the issue {issue_key} that you are facing, we will get back to you as soon as possible.
    Regards,
    Alnafi Team """

    # this format is special and you need to follow it always, if you don't use \n everything will be sent as subject 
    text = f"Subject: {SUBJECT}\n\n{MESSAGE}"


    # smtp mail of google.com, as gmail works on 587 port
    connection = smtplib.SMTP("smtp.gmail.com",587)

    # starting transport layer , TLS: transport layer security
    connection.starttls()

    # logging in the account
    connection.login(user=my_mail,password=my_pswd)

    # sending mail
    connection.sendmail(from_addr=my_mail,to_addrs=to_mail,msg=text)


    # closing the connection after sending mail
    connection.close()

else:
    print(f"The memory is good on {linux_server} and available memory is {available_memory}")
        

In [12]:
issue.key

'LP-17'